In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
AUDIO_PATH = '../data/ESC-50-master/ESC-50-master/audio/'
META_PATH = '../data/ESC-50-master/ESC-50-master/meta/esc50.csv'

df = pd.read_csv(META_PATH)

N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 50

print(f'Total clips: {len(df)}')
print(f'Classes: {df["category"].nunique()}')

Total clips: 2000
Classes: 50


In [4]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

class BirdSoundDataset(Dataset):
    def __init__(self, df, audio_path):
        self.df = df.reset_index(drop=True)
        self.audio_path = audio_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_path, row['filename'])

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

print('Dataset class defined')

Dataset class defined


In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = BirdSoundDataset(train_df, AUDIO_PATH)
test_dataset = BirdSoundDataset(test_df, AUDIO_PATH)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 1600
Test samples:  400
Train batches: 50
Test batches:  13
Batch X shape: torch.Size([32, 1, 128, 216])
Batch y shape: torch.Size([32])


In [6]:
class EchoNet(nn.Module):
    def __init__(self, num_classes=50):
        super(EchoNet, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 27, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = EchoNet(num_classes=NUM_CLASSES).to(device)
print(model)

EchoNet(
  (conv_block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=5

In [7]:
with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 216).to(device)
    out = model.conv_block1(dummy)
    print(f'After block1: {out.shape}')
    out = model.conv_block2(out)
    print(f'After block2: {out.shape}')
    out = model.conv_block3(out)
    print(f'After block3: {out.shape}')
    print(f'Flattened: {out.view(1, -1).shape[1]}')

After block1: torch.Size([1, 32, 64, 108])
After block2: torch.Size([1, 64, 32, 54])
After block3: torch.Size([1, 128, 16, 27])
Flattened: 55296


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 9.1920 | Test Acc: 0.0650
Epoch [2/30] Loss: 3.7359 | Test Acc: 0.0775
Epoch [3/30] Loss: 3.6489 | Test Acc: 0.1175
Epoch [4/30] Loss: 3.5655 | Test Acc: 0.1100
Epoch [5/30] Loss: 3.4927 | Test Acc: 0.1175
Epoch [6/30] Loss: 3.4585 | Test Acc: 0.1250
Epoch [7/30] Loss: 3.4219 | Test Acc: 0.1675
Epoch [8/30] Loss: 3.3075 | Test Acc: 0.1400
Epoch [9/30] Loss: 3.3086 | Test Acc: 0.1675
Epoch [10/30] Loss: 3.2373 | Test Acc: 0.1700
Epoch [11/30] Loss: 3.1780 | Test Acc: 0.2300
Epoch [12/30] Loss: 3.0354 | Test Acc: 0.2800
Epoch [13/30] Loss: 2.9556 | Test Acc: 0.2525
Epoch [14/30] Loss: 2.9592 | Test Acc: 0.2625
Epoch [15/30] Loss: 2.9088 | Test Acc: 0.2800
Epoch [16/30] Loss: 2.8214 | Test Acc: 0.3075
Epoch [17/30] Loss: 2.8330 | Test Acc: 0.3275
Epoch [18/30] Loss: 2.7082 | Test Acc: 0.3300
Epoch [19/30] Loss: 2.7015 | Test Acc: 0.3575
Epoch [20/30] Loss: 2.6135 | Test Acc: 0.3250
Epoch [21/30] Loss: 2.5280 | Test Acc: 0.3875
Epoch [22/30] Loss: 2.4703 | Test Acc: 0.40

In [9]:
EPOCHS2 = 30

for epoch in range(EPOCHS2):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+31}/{60}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [31/60] Loss: 2.0971 | Test Acc: 0.4425
Epoch [32/60] Loss: 2.0104 | Test Acc: 0.4875
Epoch [33/60] Loss: 2.0046 | Test Acc: 0.4325
Epoch [34/60] Loss: 1.9753 | Test Acc: 0.4775
Epoch [35/60] Loss: 1.9739 | Test Acc: 0.4500
Epoch [36/60] Loss: 1.9471 | Test Acc: 0.4850
Epoch [37/60] Loss: 1.8987 | Test Acc: 0.4850
Epoch [38/60] Loss: 1.9186 | Test Acc: 0.4850
Epoch [39/60] Loss: 1.8802 | Test Acc: 0.5050
Epoch [40/60] Loss: 1.8565 | Test Acc: 0.5225
Epoch [41/60] Loss: 1.8054 | Test Acc: 0.4950
Epoch [42/60] Loss: 1.8152 | Test Acc: 0.5125
Epoch [43/60] Loss: 1.8161 | Test Acc: 0.5200
Epoch [44/60] Loss: 1.8275 | Test Acc: 0.5100
Epoch [45/60] Loss: 1.8241 | Test Acc: 0.5125
Epoch [46/60] Loss: 1.7590 | Test Acc: 0.5025
Epoch [47/60] Loss: 1.7699 | Test Acc: 0.5200
Epoch [48/60] Loss: 1.7010 | Test Acc: 0.5025
Epoch [49/60] Loss: 1.7929 | Test Acc: 0.5050
Epoch [50/60] Loss: 1.7258 | Test Acc: 0.5225
Epoch [51/60] Loss: 1.7429 | Test Acc: 0.5275
Epoch [52/60] Loss: 1.6862 | Test 

In [10]:
torch.save(model.state_dict(), '../models/echonet_scratch.pth')
print('Model saved')

Model saved


In [13]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(efficientnet.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Arup Sarkar/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:04<00:00, 5.07MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [14]:
# Replace final layer for 50 classes
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

# Convert grayscale (1 channel) to 3 channels EfficientNet expects
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Move to GPU
efficientnet = efficientnet.to(device)

# Freeze all layers except classifier
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

# Unfreeze first conv too since we changed it
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet ready')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet ready
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=50, bias=True)
)


In [15]:
optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_eff = optim.lr_scheduler.StepLR(optimizer_eff, step_size=10, gamma=0.5)

EPOCHS_EFF = 30
eff_losses = []
eff_accuracies = []

for epoch in range(EPOCHS_EFF):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff.step()
        running_loss += loss.item()

    scheduler_eff.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    eff_losses.append(avg_loss)
    eff_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS_EFF}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 3.4648 | Test Acc: 0.3950
Epoch [2/30] Loss: 2.4775 | Test Acc: 0.4875
Epoch [3/30] Loss: 1.9452 | Test Acc: 0.5150
Epoch [4/30] Loss: 1.6036 | Test Acc: 0.5625
Epoch [5/30] Loss: 1.3937 | Test Acc: 0.5450
Epoch [6/30] Loss: 1.2138 | Test Acc: 0.5725
Epoch [7/30] Loss: 1.0809 | Test Acc: 0.5775
Epoch [8/30] Loss: 0.9984 | Test Acc: 0.5800
Epoch [9/30] Loss: 0.9172 | Test Acc: 0.5850
Epoch [10/30] Loss: 0.8492 | Test Acc: 0.6150
Epoch [11/30] Loss: 0.7731 | Test Acc: 0.6100
Epoch [12/30] Loss: 0.7278 | Test Acc: 0.6250
Epoch [13/30] Loss: 0.7168 | Test Acc: 0.6050
Epoch [14/30] Loss: 0.6767 | Test Acc: 0.6025
Epoch [15/30] Loss: 0.6513 | Test Acc: 0.6225
Epoch [16/30] Loss: 0.6430 | Test Acc: 0.6250
Epoch [17/30] Loss: 0.6165 | Test Acc: 0.6200
Epoch [18/30] Loss: 0.5767 | Test Acc: 0.6125
Epoch [19/30] Loss: 0.5855 | Test Acc: 0.6300
Epoch [20/30] Loss: 0.5507 | Test Acc: 0.6200
Epoch [21/30] Loss: 0.5384 | Test Acc: 0.6250
Epoch [22/30] Loss: 0.5179 | Test Acc: 0.61

In [16]:
# Unfreeze everything
for param in efficientnet.parameters():
    param.requires_grad = True

# Lower learning rate for fine-tuning
optimizer_eff2 = optim.Adam(efficientnet.parameters(), lr=0.0001)
scheduler_eff2 = optim.lr_scheduler.StepLR(optimizer_eff2, step_size=10, gamma=0.5)

EPOCHS_EFF2 = 20
for epoch in range(EPOCHS_EFF2):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff2.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff2.step()
        running_loss += loss.item()

    scheduler_eff2.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+31}/{50}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')


Epoch [31/50] Loss: 0.4268 | Test Acc: 0.6625
Epoch [32/50] Loss: 0.2875 | Test Acc: 0.6875
Epoch [33/50] Loss: 0.1902 | Test Acc: 0.7000
Epoch [34/50] Loss: 0.1469 | Test Acc: 0.7025
Epoch [35/50] Loss: 0.1169 | Test Acc: 0.6975
Epoch [36/50] Loss: 0.0973 | Test Acc: 0.7025
Epoch [37/50] Loss: 0.0700 | Test Acc: 0.7050
Epoch [38/50] Loss: 0.0743 | Test Acc: 0.7175
Epoch [39/50] Loss: 0.0599 | Test Acc: 0.7350
Epoch [40/50] Loss: 0.0557 | Test Acc: 0.7450
Epoch [41/50] Loss: 0.0450 | Test Acc: 0.7475
Epoch [42/50] Loss: 0.0440 | Test Acc: 0.7375
Epoch [43/50] Loss: 0.0400 | Test Acc: 0.7425
Epoch [44/50] Loss: 0.0316 | Test Acc: 0.7400
Epoch [45/50] Loss: 0.0328 | Test Acc: 0.7425
Epoch [46/50] Loss: 0.0325 | Test Acc: 0.7450
Epoch [47/50] Loss: 0.0271 | Test Acc: 0.7350
Epoch [48/50] Loss: 0.0245 | Test Acc: 0.7350
Epoch [49/50] Loss: 0.0199 | Test Acc: 0.7450
Epoch [50/50] Loss: 0.0181 | Test Acc: 0.7450


In [17]:
torch.save(efficientnet.state_dict(), '../models/echonet_efficientnet.pth')
print('Model saved')

Model saved
